In [2]:
import sys
print(sys.executable)

c:\Users\EAG28\OneDrive\Документы\2026W\Project\ev-traffic-charging-prediction\.venv\Scripts\python.exe


In [3]:
import pandas as pd

In [4]:
nodes = pd.read_csv("../data/raw/shenzhen/node_shenzhen.csv")
edges = pd.read_csv("../data/raw/shenzhen/edge_shenzhen.csv")

trajectories = pd.read_csv("../data/raw/shenzhen/traj_shenzhen_20201104.csv")

In [5]:
display(nodes.head())
display(edges.head())
display(trajectories.head())

print("Nodes:", nodes.shape)
print("Edges:", edges.shape)
print("Trajectories:", trajectories.shape)

,NodeID,Longitude,Latitude,HasCamera
0,0,114.049332,22.690314,0
1,1,114.048449,22.675017,0
2,2,114.046650,22.685071,0
3,3,114.048557,22.676590,0
4,4,114.050360,22.681492,0


,Origin,Destination,Class,Geometry,Length
0,0,1,motorway,114.049332-22.690314_114.048995-22.688018_114....,1699.3
1,0,2,motorway_link,114.049332-22.690314_114.049234-22.689926_114....,663.9
2,1,10107,motorway,114.048449-22.675017_114.048508-22.673645,152.0
3,2,8,motorway_link,114.04665-22.685071_114.046193-22.684735_114.0...,366.4
4,2,7,motorway_link,114.04665-22.685071_114.046416-22.684763_114.0...,1202.3


,VehicleID,TripID,Points,DepartureTime,Duration,Length
0,0,0,1226-41543.0_11706-41546.2_1227-41552.5_10226-...,41543.0,1165.0,6764.6
1,0,1,11699-62360.0_1495-62374.7_10227-62571.3_10225...,62360.0,419.0,2455.3
2,1,0,11699-32737.0_1495-32742.6_10227-32805.0_10225...,32737.0,5680.0,19380.9
3,1,1,3829-60593.0_8182-60624.4_8181-60633.1_3838-60...,60593.0,1921.5,5086.3
4,1,2,11699-64357.5_1495-64371.4_10227-64556.9_10225...,64357.5,545.5,2455.3


Nodes: (11933, 4)
Edges: (27410, 5)
Trajectories: (569057, 6)


Exploring one trajectory

In [6]:
points = trajectories.iloc[0]["Points"]

print(points)

1226-41543.0_11706-41546.2_1227-41552.5_10226-41560.1_11320-41574.3_11699-41608.0_1188-41620.6_1190-41640.4_4209-41667.0_1246-41784.9_1255-41929.7_1254-41984.6_10261-42006.0_1253-42031.0_1256-42136.2_1258-42178.4_8954-42216.7_8958-42294.9_4544-42459.3_4547-42504.1_4550-42539.1_11775-42554.4_6851-42561.9_9204-42625.5_6851-42700.6_11775-42708.0


In [7]:
# parsing the trajectory
parsed_points = [
    (int(point.split("-")[0]), float(point.split("-")[1]))
    for point in points.split("_")
]

parsed_points

[(1226, 41543.0),
 (11706, 41546.2),
 (1227, 41552.5),
 (10226, 41560.1),
 (11320, 41574.3),
 (11699, 41608.0),
 (1188, 41620.6),
 (1190, 41640.4),
 (4209, 41667.0),
 (1246, 41784.9),
 (1255, 41929.7),
 (1254, 41984.6),
 (10261, 42006.0),
 (1253, 42031.0),
 (1256, 42136.2),
 (1258, 42178.4),
 (8954, 42216.7),
 (8958, 42294.9),
 (4544, 42459.3),
 (4547, 42504.1),
 (4550, 42539.1),
 (11775, 42554.4),
 (6851, 42561.9),
 (9204, 42625.5),
 (6851, 42700.6),
 (11775, 42708.0)]

In [8]:
print("Number of points:", len(parsed_points))
print("First 10 points:")
parsed_points[:10]

Number of points: 26
First 10 points:


[(1226, 41543.0),
 (11706, 41546.2),
 (1227, 41552.5),
 (10226, 41560.1),
 (11320, 41574.3),
 (11699, 41608.0),
 (1188, 41620.6),
 (1190, 41640.4),
 (4209, 41667.0),
 (1246, 41784.9)]

In [9]:
node_pairs = [
    (parsed_points[i][0], parsed_points[i + 1][0])
    for i in range(len(parsed_points) - 1)
]

node_pairs[:10]

[(1226, 11706),
 (11706, 1227),
 (1227, 10226),
 (10226, 11320),
 (11320, 11699),
 (11699, 1188),
 (1188, 1190),
 (1190, 4209),
 (4209, 1246),
 (1246, 1255)]

Veryfying whether these are real node edges

In [10]:
edge_pairs = set(zip(edges["Origin"], edges["Destination"]))

edge_check = [
    (origin, destination, (origin, destination) in edge_pairs)
    for origin, destination in node_pairs
]

edge_check[:10]

[(1226, 11706, True),
 (11706, 1227, True),
 (1227, 10226, True),
 (10226, 11320, True),
 (11320, 11699, True),
 (11699, 1188, True),
 (1188, 1190, True),
 (1190, 4209, True),
 (4209, 1246, True),
 (1246, 1255, True)]

In [11]:
all_edges_valid = all(is_valid for _, _, is_valid in edge_check)

print("All consecutive nodes are valid edges:", all_edges_valid)
print("Valid edges:", sum(is_valid for _, _, is_valid in edge_check))
print("Total transitions:", len(edge_check))

All consecutive nodes are valid edges: True
Valid edges: 25
Total transitions: 25


Visualising the trajectory via folium package

In [12]:
import folium

# Fast lookup: NodeID -> (latitude, longitude)
node_coordinates = nodes.set_index("NodeID")[["Latitude", "Longitude"]]

# Convert our trajectory node IDs into map coordinates
route_coordinates = [
    [
        node_coordinates.loc[node_id, "Latitude"],
        node_coordinates.loc[node_id, "Longitude"],
    ]
    for node_id, _ in parsed_points
]

route_coordinates[:5]

[[np.float64(22.634135), np.float64(114.032018)],
 [np.float64(22.634049), np.float64(114.032194)],
 [np.float64(22.634021), np.float64(114.032614)],
 [np.float64(22.633586), np.float64(114.033572)],
 [np.float64(22.633286), np.float64(114.036184)]]

In [13]:
route_map = folium.Map(
    location=route_coordinates[0],
    zoom_start=13,
    tiles=None,
)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Esri",
    name="Esri World Street Map",
).add_to(route_map)

folium.PolyLine(
    route_coordinates,
    weight=5,
).add_to(route_map)

folium.Marker(
    route_coordinates[0],
    tooltip="Start",
).add_to(route_map)

folium.Marker(
    route_coordinates[-1],
    tooltip="End",
).add_to(route_map)

route_map.save("../route_map.html")


Trying to follow road geometry

In [14]:
print(edges.iloc[0]["Geometry"])

114.049332-22.690314_114.048995-22.688018_114.048866-22.687063_114.048738-22.685958_114.048635-22.685032_114.048537-22.683722_114.048481-22.68278_114.048395-22.681357_114.048369-22.680485_114.048377-22.678973_114.048394-22.677271_114.048449-22.675017


In [15]:
edge_geometries = {
    (row.Origin, row.Destination): row.Geometry
    for row in edges.itertuples()
}

In [16]:
edge_geometries[(1226, 11706)]

'114.032018-22.634135_114.032201-22.634134_114.032194-22.634049'

In [17]:
def parse_geometry(geometry):
    return [
        [float(point.split("-")[1]), float(point.split("-")[0])]
        for point in geometry.split("_")
    ]

In [18]:
parse_geometry(edge_geometries[(1226, 11706)])[:5]

[[22.634135, 114.032018], [22.634134, 114.032201], [22.634049, 114.032194]]

In [19]:
# create a new map

road_map = folium.Map(
    location=route_coordinates[0],
    zoom_start=13,
    tiles=None,
)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Esri",
    name="Esri World Street Map",
).add_to(road_map)

In [20]:
# draw an actual edge

for origin, destination in node_pairs:
    geometry = edge_geometries[(origin, destination)]

    folium.PolyLine(
        parse_geometry(geometry),
        weight=5,
    ).add_to(road_map)

In [21]:
# adding start and end
folium.Marker(
    route_coordinates[0],
    tooltip="Start",
).add_to(road_map)

folium.Marker(
    route_coordinates[-1],
    tooltip="End",
).add_to(road_map)

road_map.save("../road_geometry_map.html")

Extracting intersection movements for one trajection

In [22]:
movements = []

for i in range(1, len(parsed_points) - 1):
    incoming_node = parsed_points[i - 1][0]
    intersection = parsed_points[i][0]
    outgoing_node = parsed_points[i + 1][0]
    time = parsed_points[i][1]

    movements.append(
        {
            "IncomingNode": incoming_node,
            "Intersection": intersection,
            "OutgoingNode": outgoing_node,
            "Time": time,
        }
    )

In [23]:
movements_df = pd.DataFrame(movements)

movements_df.head(10)

,IncomingNode,Intersection,OutgoingNode,Time
0,1226,11706,1227,41546.2
1,11706,1227,10226,41552.5
2,1227,10226,11320,41560.1
3,10226,11320,11699,41574.3
4,11320,11699,1188,41608.0
5,11699,1188,1190,41620.6
6,1188,1190,4209,41640.4
7,1190,4209,1246,41667.0
8,4209,1246,1255,41784.9
9,1246,1255,1254,41929.7


In [24]:
# time format change

movements_df["TimeOfDay"] = (
    pd.to_datetime(movements_df["Time"], unit="s")
    .dt.strftime("%H:%M:%S")
)

movements_df["Hour"] = (movements_df["Time"] // 3600).astype(int)

movements_df.head(10)

,IncomingNode,Intersection,OutgoingNode,Time,TimeOfDay,Hour
0,1226,11706,1227,41546.2,11:32:26,11
1,11706,1227,10226,41552.5,11:32:32,11
2,1227,10226,11320,41560.1,11:32:40,11
3,10226,11320,11699,41574.3,11:32:54,11
4,11320,11699,1188,41608.0,11:33:28,11
5,11699,1188,1190,41620.6,11:33:40,11
6,1188,1190,4209,41640.4,11:34:00,11
7,1190,4209,1246,41667.0,11:34:27,11
8,4209,1246,1255,41784.9,11:36:24,11
9,1246,1255,1254,41929.7,11:38:49,11


Testing the trajectories parcer

In [25]:
from ev_traffic.trajectories import extract_movements, parse_trajectory

test_points = trajectories.iloc[0]["Points"]

parsed = parse_trajectory(test_points)
movements = extract_movements(parsed)

len(parsed), len(movements)

(26, 24)

In [26]:
movements[:5]

[(1226, 11706, 1227, 41546.2),
 (11706, 1227, 10226, 41552.5),
 (1227, 10226, 11320, 41560.1),
 (10226, 11320, 11699, 41574.3),
 (11320, 11699, 1188, 41608.0)]

Extracting movements from multiple trajectories

In [27]:
movement_records = []

for row in trajectories.head(1000).itertuples(index=False):
    parsed = parse_trajectory(row.Points)
    trip_movements = extract_movements(parsed)

    for incoming, intersection, outgoing, time in trip_movements:
        movement_records.append(
            {
                "VehicleID": row.VehicleID,
                "TripID": row.TripID,
                "IncomingNode": incoming,
                "Intersection": intersection,
                "OutgoingNode": outgoing,
                "Time": time,
            }
        )

movements_sample = pd.DataFrame(movement_records)

movements_sample.head()

,VehicleID,TripID,IncomingNode,Intersection,OutgoingNode,Time
0,0,0,1226,11706,1227,41546.2
1,0,0,11706,1227,10226,41552.5
2,0,0,1227,10226,11320,41560.1
3,0,0,10226,11320,11699,41574.3
4,0,0,11320,11699,1188,41608.0


In [28]:
movements_sample.shape

(28607, 6)

In [29]:
movements_sample[["VehicleID", "TripID"]].drop_duplicates().shape

(972, 2)

In [30]:
movements_sample.isna().sum()

VehicleID       0
TripID          0
IncomingNode    0
Intersection    0
OutgoingNode    0
Time            0
dtype: int64

In [31]:
# searching for missing trajectories

point_counts = (
    trajectories.head(1000)["Points"]
    .str.count("_")
    .add(1)
)

point_counts.value_counts().sort_index().head(10)

Points
2     26
3     41
4     29
5     52
6     36
7     37
8     27
9     27
10    31
11    28
Name: count, dtype: int64

In [32]:
(point_counts < 3).sum()

np.int64(26)

1000 input trajectories:
- 974 trajectories with ≥3 nodes → can define at least one A → B → C movement
- 26 trajectories with only 2 nodes → no intersection movement possible

Validation  of road connections

In [33]:
# set of all directed road connections
edge_pairs = set(zip(edges["Origin"], edges["Destination"], strict=True))

len(edge_pairs), len(edges)

(27410, 27410)

In [34]:
incoming_edges_valid = [
    (incoming, intersection) in edge_pairs
    for incoming, intersection in zip(
        movements_sample["IncomingNode"],
        movements_sample["Intersection"],
        strict=True,
    )
]

outgoing_edges_valid = [
    (intersection, outgoing) in edge_pairs
    for intersection, outgoing in zip(
        movements_sample["Intersection"],
        movements_sample["OutgoingNode"],
        strict=True,
    )
]

sum(incoming_edges_valid), sum(outgoing_edges_valid), len(movements_sample)

(28607, 28607, 28607)

In [35]:
# test on missing nodes
invalid_incoming = movements_sample[
    ~pd.Series(incoming_edges_valid, index=movements_sample.index)
]

invalid_outgoing = movements_sample[
    ~pd.Series(outgoing_edges_valid, index=movements_sample.index)
]

invalid_incoming.head(10)

,VehicleID,TripID,IncomingNode,Intersection,OutgoingNode,Time


In [36]:
invalid_outgoing.head(10)

,VehicleID,TripID,IncomingNode,Intersection,OutgoingNode,Time


Found duplicate nodes

In [37]:
duplicate_node_movements = movements_sample[
    (movements_sample["IncomingNode"] == movements_sample["Intersection"])
    | (movements_sample["Intersection"] == movements_sample["OutgoingNode"])
]

duplicate_node_movements.shape

(0, 6)

In [38]:
duplicate_node_movements.head(10)

,VehicleID,TripID,IncomingNode,Intersection,OutgoingNode,Time


In [39]:
invalid_incoming_non_duplicate = invalid_incoming[
    invalid_incoming["IncomingNode"] != invalid_incoming["Intersection"]
]

invalid_outgoing_non_duplicate = invalid_outgoing[
    invalid_outgoing["Intersection"] != invalid_outgoing["OutgoingNode"]
]

len(invalid_incoming_non_duplicate), len(invalid_outgoing_non_duplicate)

(0, 0)

Theory proved correct and the nodes are indeed duplicates and there needs to be a correction on fake movements. Needs to be an exclusion of duplicate nodes. Corrections to trajectories.py. Will remove only consecutive duplicates, since if car genuinly travels in circles it's not a duplicate.

Done - 0 duplicates.

## Road classes and highway exits

Inspecting the road classes and network structure to understand how intersections
and highway exits can be distinguished

In [40]:
edges["Class"].value_counts(dropna=False)

Class
residential       7597
tertiary          6166
secondary         3779
primary           3535
unclassified      3348
primary_link       924
trunk              471
motorway_link      414
trunk_link         379
secondary_link     279
motorway           224
living_street      209
tertiary_link       85
Name: count, dtype: int64

In [41]:
highway_classes = ["motorway", "motorway_link", "trunk", "trunk_link"]

edges[edges["Class"].isin(highway_classes)]["Class"].value_counts()

Class
trunk            471
motorway_link    414
trunk_link       379
motorway         224
Name: count, dtype: int64

In [42]:
highway_edges = edges[edges["Class"].isin(highway_classes)].copy()

highway_edges.head(10)

,Origin,Destination,Class,Geometry,Length
0,0,1,motorway,114.049332-22.690314_114.048995-22.688018_114....,1699.3
1,0,2,motorway_link,114.049332-22.690314_114.049234-22.689926_114....,663.9
2,1,10107,motorway,114.048449-22.675017_114.048508-22.673645,152.0
3,2,8,motorway_link,114.04665-22.685071_114.046193-22.684735_114.0...,366.4
4,2,7,motorway_link,114.04665-22.685071_114.046416-22.684763_114.0...,1202.3
5,3,4,motorway_link,114.048557-22.67659_114.04858-22.677141_114.04...,600.3
6,3,5,motorway,114.048557-22.67659_114.04854-22.677263_114.04...,1733.7
7,4,6,motorway_link,114.05036-22.681492_114.050678-22.681798_114.0...,1133.1
8,4,7,motorway_link,114.05036-22.681492_114.050815-22.681816_114.0...,348.8
9,5,11895,motorway,114.049885-22.692168_114.049963-22.692642_114....,76.0


It seems like at node 0, a vehicle may have a choice between going forward or entering a motorway-link edge. Looks like an exit/divergence point

In [43]:
candidate_nodes = [0, 2, 3, 4]

edges[edges["Origin"].isin(candidate_nodes)].sort_values(
    ["Origin", "Class", "Destination"]
)[["Origin", "Destination", "Class", "Length"]]

,Origin,Destination,Class,Length
0,0,1,motorway,1699.3
1,0,2,motorway_link,663.9
4,2,7,motorway_link,1202.3
3,2,8,motorway_link,366.4
6,3,5,motorway,1733.7
5,3,4,motorway_link,600.3
7,4,6,motorway_link,1133.1
8,4,7,motorway_link,348.8


origins 0 and 3 seem to be candidates for highway divergence points (lead both forward along motorway and motorway links)
next: what road did the vehicle arrive on?

In [44]:
candidate_nodes = [0, 3]

for node in candidate_nodes:
    print(f"\nNode {node}")

    print("Incoming:")
    display(
        edges[edges["Destination"] == node][
            ["Origin", "Destination", "Class", "Length"]
        ]
    )

    print("Outgoing:")
    display(
        edges[edges["Origin"] == node][
            ["Origin", "Destination", "Class", "Length"]
        ]
    )


Node 0
Incoming:


,Origin,Destination,Class,Length
27373,11914,0,motorway,807.3


Outgoing:


,Origin,Destination,Class,Length
0,0,1,motorway,1699.3
1,0,2,motorway_link,663.9



Node 3
Incoming:


,Origin,Destination,Class,Length
2156,1093,3,motorway,313.0


Outgoing:


,Origin,Destination,Class,Length
5,3,4,motorway_link,600.3
6,3,5,motorway,1733.7


In [46]:
# finsing all candidates for the highway decision points
motorway_exit_candidates = []

for node in edges["Origin"].unique():
    incoming_classes = set(
        edges.loc[edges["Destination"] == node, "Class"]
    )
    outgoing_classes = set(
        edges.loc[edges["Origin"] == node, "Class"]
    )

    if (
        "motorway" in incoming_classes
        and "motorway" in outgoing_classes
        and "motorway_link" in outgoing_classes
    ):
        motorway_exit_candidates.append(node)

print("Number of motorway exit candidates:", len(motorway_exit_candidates))

Number of motorway exit candidates: 80


In [47]:
sample_candidates = motorway_exit_candidates[::10]

for node in sample_candidates:
    incoming = edges[edges["Destination"] == node]
    outgoing = edges[edges["Origin"] == node]

    print(f"\nNode {node}")
    print(
        "Incoming:",
        list(zip(incoming["Class"], incoming["Origin"])),
    )
    print(
        "Outgoing:",
        list(zip(outgoing["Class"], outgoing["Destination"])),
    )


Node 0
Incoming: [('motorway', 11914)]
Outgoing: [('motorway', 1), ('motorway_link', 2)]

Node 487
Incoming: [('motorway', 9994)]
Outgoing: [('motorway_link', 11916), ('motorway', 11917)]

Node 1094
Incoming: [('motorway', 8885)]
Outgoing: [('motorway_link', 75), ('motorway', 1095)]

Node 3049
Incoming: [('motorway', 11828)]
Outgoing: [('motorway', 3050), ('motorway_link', 3051)]

Node 3434
Incoming: [('motorway', 10297)]
Outgoing: [('motorway', 3435), ('motorway_link', 3436)]

Node 3696
Incoming: [('motorway', 9528)]
Outgoing: [('motorway_link', 3034), ('motorway', 3037)]

Node 6399
Incoming: [('motorway', 6398)]
Outgoing: [('motorway_link', 6409), ('motorway', 6410)]

Node 7952
Incoming: [('motorway', 3263)]
Outgoing: [('motorway_link', 7949), ('motorway', 7953)]


are trunk roads also highways or only motorways?

In [48]:
trunk_exit_candidates = []

for node in edges["Origin"].unique():
    incoming_classes = set(
        edges.loc[edges["Destination"] == node, "Class"]
    )
    outgoing_classes = set(
        edges.loc[edges["Origin"] == node, "Class"]
    )

    if (
        "trunk" in incoming_classes
        and "trunk" in outgoing_classes
        and "trunk_link" in outgoing_classes
    ):
        trunk_exit_candidates.append(node)

print("Number of trunk exit candidates:", len(trunk_exit_candidates))
print("First 20:", trunk_exit_candidates[:20])

Number of trunk exit candidates: 116
First 20: [np.int64(17), np.int64(33), np.int64(50), np.int64(53), np.int64(56), np.int64(71), np.int64(93), np.int64(96), np.int64(100), np.int64(129), np.int64(162), np.int64(197), np.int64(323), np.int64(360), np.int64(361), np.int64(364), np.int64(431), np.int64(437), np.int64(440), np.int64(456)]


In [49]:
sample_trunk_candidates = trunk_exit_candidates[::15]

for node in sample_trunk_candidates:
    incoming = edges[edges["Destination"] == node]
    outgoing = edges[edges["Origin"] == node]

    print(f"\nNode {node}")
    print(
        "Incoming:",
        list(zip(incoming["Class"], incoming["Origin"])),
    )
    print(
        "Outgoing:",
        list(zip(outgoing["Class"], outgoing["Destination"])),
    )


Node 17
Incoming: [('trunk', 16)]
Outgoing: [('trunk_link', 18), ('trunk', 11760)]

Node 364
Incoming: [('trunk', 8272)]
Outgoing: [('trunk_link', 365), ('trunk', 366)]

Node 789
Incoming: [('trunk', 7021)]
Outgoing: [('trunk_link', 790), ('trunk', 791)]

Node 921
Incoming: [('trunk', 917)]
Outgoing: [('trunk', 6078), ('trunk_link', 919)]

Node 1106
Incoming: [('trunk', 1111), ('residential', 3684), ('primary', 3685)]
Outgoing: [('trunk_link', 3683), ('trunk', 1071), ('residential', 3684)]

Node 3033
Incoming: [('trunk', 3032)]
Outgoing: [('trunk', 4375), ('trunk_link', 4377)]

Node 4338
Incoming: [('trunk', 2635)]
Outgoing: [('trunk', 8270), ('trunk_link', 11662)]

Node 8276
Incoming: [('trunk', 3828)]
Outgoing: [('trunk_link', 8275), ('trunk', 11662)]


##### Potential Highway logic:

incoming Class = trunk/motorway -> decision node ->

outgoing same main-road class → CONTINUE **OR** outgoing corresponding *_link → EXIT